<a href="https://colab.research.google.com/github/marcossouza007/Ciencia_de_Dados/blob/main/Sono_e_Produtividade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

def ler_csv(caminho):

    with open(caminho, "r", encoding="utf-8") as f:
        linhas = f.readlines()

    cabecalho = linhas[0].strip().split(";")
    registros = []

    for linha in linhas[1:]:
        valores = linha.strip().split(";")

        registro = {}
        for i in range(len(cabecalho)):
            if i < len(valores):
                registro[cabecalho[i]] = valores[i]
            else:
                registro[cabecalho[i]] = ""

        registros.append(registro)

    return registros


# VALIDAÇÃO
# Remove registros sem horas de sono válidas

def validar_registros(registros):

    validos = []
    invalidos = []

    for r in registros:
        try:
            if r["horas_sono"] == "":
                invalidos.append(r)
            else:
                float(r["horas_sono"])
                validos.append(r)
        except:
            invalidos.append(r)

    return validos, invalidos


# LIMPEZA E PADRONIZAÇÃO

def limpar(registros):

    for r in registros:
        for k in r:
            r[k] = r[k].strip()

    return registros


# ESTATÍSTICAS NUMÉRICAS

def estatisticas(registros):

    sono = []
    produtivas = []

    for r in registros:
        try:
            sono.append(float(r["horas_sono"]))
            produtivas.append(float(r["horas_produtivas"]))
        except:
            pass

    media_sono = sum(sono) / len(sono)
    media_prod = sum(produtivas) / len(produtivas)

    return {
        "media_sono": media_sono,
        "min_sono": min(sono),
        "max_sono": max(sono),
        "media_produtiva": media_prod,
        "total_registros": len(registros)
    }



# CONTAGEM DE CATEGORIAS

def contagem(registros, campo):

    cont = {}

    for r in registros:
        valor = r[campo]
        cont[valor] = cont.get(valor, 0) + 1

    return cont



# RANKING POR HORAS DE SONO

def ranking_sono(registros, top=5):

    ordenado = sorted(
        registros,
        key=lambda r: float(r["horas_sono"]),
        reverse=True
    )

    return ordenado[:top]



# FILTRO DINÂMICO

def filtrar(registros, campo, valor):

    return [r for r in registros if r[campo] == valor]


# BUSCA TEXTUAL

def busca_texto(registros, termo):

    termo = termo.lower()
    encontrados = []

    for r in registros:
        for v in r.values():
            if termo in v.lower():
                encontrados.append(r)
                break

    return encontrados



# DETECÇÃO DE INCONSISTÊNCIA
# Pouco sono + alta produtividade

def inconsistencias(registros):

    inc = []

    for r in registros:
        try:
            sono = float(r["horas_sono"])
            prod = float(r["produtividade_boa_noite"])

            if sono < 4 and prod >= 8:
                inc.append(r)
        except:
            pass

    return inc



# GERAR RELATÓRIO TEXTUAL AUTOMÁTICO

def gerar_relatorio(stats, ocupacoes, telas, ranking, inconsist):

    texto = ""

    texto += "RELATÓRIO — SONO E PRODUTIVIDADE\n"
    texto += "=================================\n\n"

    texto += "1. Estatísticas Gerais\n"
    texto += f"Média de horas de sono: {stats['media_sono']:.2f}\n"
    texto += f"Mínimo de horas de sono: {stats['min_sono']}\n"
    texto += f"Máximo de horas de sono: {stats['max_sono']}\n"
    texto += f"Média de horas produtivas: {stats['media_produtiva']:.2f}\n"
    texto += f"Total de participantes: {stats['total_registros']}\n\n"

    texto += "2. Distribuição por Ocupação\n"
    for k, v in ocupacoes.items():
        texto += f"{k}: {v}\n"

    texto += "\n3. Uso de telas antes de dormir\n"
    for k, v in telas.items():
        texto += f"{k}: {v}\n"

    texto += "\n4. Ranking — Maiores horas de sono\n"
    for i, r in enumerate(ranking, 1):
        texto += f"{i}º lugar: {r['horas_sono']} horas — {r['ocupacao']}\n"

    texto += "\n5. Inconsistências encontradas\n"
    texto += f"Total: {len(inconsist)} registros\n"

    return texto



# PROGRAMA PRINCIPAL


caminho = "/content/drive/MyDrive/sono_produtividade.csv"

dados = ler_csv(caminho)

validos, invalidos = validar_registros(dados)

validos = limpar(validos)

stats = estatisticas(validos)

ocupacoes = contagem(validos, "ocupacao")
telas = contagem(validos, "uso_telas")

ranking = ranking_sono(validos)

inc = inconsistencias(validos)

relatorio = gerar_relatorio(stats, ocupacoes, telas, ranking, inc)

print(relatorio)


# Salvar relatório no Drive
with open("/content/drive/MyDrive/relatorio_sono.txt", "w", encoding="utf-8") as f:
    f.write(relatorio)

RELATÓRIO — SONO E PRODUTIVIDADE

1. Estatísticas Gerais
Média de horas de sono: 6.81
Mínimo de horas de sono: 4.5
Máximo de horas de sono: 9.0
Média de horas produtivas: 6.06
Total de participantes: 45

2. Distribuição por Ocupação
Estudante: 14
Estudante e trabalhador(a): 8
Trabalhador(a) CLT: 15
Autônomo(a): 7
Outra: 1

3. Uso de telas antes de dormir
Sim: 25
Não: 20

4. Ranking — Maiores horas de sono
1º lugar: 9.0 horas — Autônomo(a)
2º lugar: 9.0 horas — Estudante
3º lugar: 8.5 horas — Autônomo(a)
4º lugar: 8.5 horas — Trabalhador(a) CLT
5º lugar: 8.0 horas — Autônomo(a)

5. Inconsistências encontradas
Total: 0 registros

